# Lecture 17 · Contrastive learning · a toy you can build in 50 lines

**Goal** · feel what contrastive self-supervision does, on a dataset you can see.

Setup · generate 3 clusters of 2D points. 'Augment' each point with small random jitter. Train a 2D → 2D encoder with InfoNCE such that jittered pairs are close, other points are far.

Watch the embedding space organize itself · same cluster → tight clump, different clusters → far apart. All without class labels!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(0); np.random.seed(0)

## 1 · Data · three clusters of 2D points

In [ ]:
N = 60
centers = torch.tensor([[-2., 0.], [2., 0.], [0., 2.5]])
cluster = torch.randint(0, 3, (N,))
x = centers[cluster] + 0.4 * torch.randn(N, 2)

# show
colors = ['#d97757', '#7b9e89', '#4a6670']
plt.figure(figsize=(5,5))
for c in range(3):
    plt.scatter(x[cluster==c, 0], x[cluster==c, 1], color=colors[c], label=f'cluster {c}')
plt.legend(); plt.title('raw 2D data · 3 clusters'); plt.axis('equal'); plt.show()

## 2 · Augmentation · small random jitter

The 'same image, different augmentation' pair — here just add noise to each point.

In [ ]:
def augment(x, sigma=0.2):
    return x + sigma * torch.randn_like(x)

## 3 · Encoder · 2D → 2D MLP

In [ ]:
class Encoder(nn.Module):
    def __init__(self, d=2, h=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(),
            nn.Linear(h, d),
        )
    def forward(self, x):
        z = self.net(x)
        # L2 normalize · points on unit circle
        return F.normalize(z, dim=-1)

model = Encoder()

## 4 · InfoNCE loss · pull positives, push negatives

In [ ]:
def infonce(z1, z2, tau=0.1):
    # z1, z2: (B, d) · aligned pairs
    B = z1.shape[0]
    # similarity matrix · (B, B)
    logits = (z1 @ z2.T) / tau
    # positives are on the diagonal
    labels = torch.arange(B)
    return F.cross_entropy(logits, labels)

## 5 · Train · watch the embedding organize

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=5e-3)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
snapshots = [0, 50, 200, 800]
ax_idx = 0

for step in range(801):
    v1 = augment(x)
    v2 = augment(x)
    z1 = model(v1)
    z2 = model(v2)
    loss = infonce(z1, z2)
    opt.zero_grad(); loss.backward(); opt.step()

    if step in snapshots:
        with torch.no_grad():
            z_all = model(x)
        for c in range(3):
            axes[ax_idx].scatter(z_all[cluster==c, 0], z_all[cluster==c, 1], color=colors[c], s=40)
        axes[ax_idx].set_title(f'step {step}  loss {loss.item():.2f}')
        # draw unit circle
        theta = np.linspace(0, 2*np.pi, 100)
        axes[ax_idx].plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3)
        axes[ax_idx].set_aspect('equal'); axes[ax_idx].set_xlim(-1.2, 1.2); axes[ax_idx].set_ylim(-1.2, 1.2)
        ax_idx += 1

plt.tight_layout(); plt.show()

**Observe** · by step 800, the three clusters have pushed to different regions of the unit circle. The network has learned **cluster structure** with no labels.

That's contrastive self-supervision in 50 lines.

## 6 · Try it · what breaks the trick?

- Set `sigma=0` in augment · no jitter. What happens? (Hint · the positive pairs are identical — trivial lookup, no structure learned.)
- Set `tau=1.0` · softer softmax. Slower learning, can it still work?
- Add a 4th cluster. Does the model adapt?

This is the pedagogical value of building it from scratch · you can break every knob and feel the dependency.